In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install datasets pyspark

1- Load 3 versions of the Arabic abstracts dataset.

2-convert them to DataFrames.

3- label each with its generation method.

4-merge them into one dataset for analysis.


In [3]:
import pandas as pd
from datasets import load_dataset


ds1 = load_dataset("KFUPM-JRCAI/arabic-generated-abstracts", split='by_polishing')
ds2 = load_dataset("KFUPM-JRCAI/arabic-generated-abstracts", split='from_title')
ds3 = load_dataset("KFUPM-JRCAI/arabic-generated-abstracts", split='from_title_and_content')

df1, df2, df3 = ds1.to_pandas(), ds2.to_pandas(), ds3.to_pandas()
df1['method'], df2['method'], df3['method'] = 'polishing', 'title', 'content'
full_df = pd.concat([df1, df2, df3], ignore_index=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/by_polishing-00000-of-00001.parquet:   0%|          | 0.00/8.49M [00:00<?, ?B/s]

data/from_title-00000-of-00001.parquet:   0%|          | 0.00/6.90M [00:00<?, ?B/s]

data/from_title_and_content-00000-of-000(…):   0%|          | 0.00/7.01M [00:00<?, ?B/s]

Generating by_polishing split:   0%|          | 0/2851 [00:00<?, ? examples/s]

Generating from_title split:   0%|          | 0/2963 [00:00<?, ? examples/s]

Generating from_title_and_content split:   0%|          | 0/2574 [00:00<?, ? examples/s]

 Save the merged dataset to Drive


In [5]:
path = "/content/drive/MyDrive/MSBDA_801_Project/data/raw/arabic_abstracts_merged.csv"
full_df.to_csv(path, index=False)
print("save in Drive")

save in Drive


**Part 2: EDA**

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

spark = SparkSession.builder \
    .appName("Arabic_AI_EDA") \
    .getOrCreate()

Load the saved CSV from Drive into Spark DataFrame

In [7]:
file_path = "/content/drive/MyDrive/MSBDA_801_Project/data/raw/arabic_abstracts_merged.csv"

df = spark.read.csv(file_path, header=True, inferSchema=True)
print("Load data into spark done ")

Load data into spark done 


Preview schema and sample rows of the dataset

In [8]:
print("Schema")
df.printSchema()

print("\n Sample")
df.show(3, truncate=40)

Schema
root
 |-- original_abstract: string (nullable = true)
 |-- allam_generated_abstract: string (nullable = true)
 |-- jais_generated_abstract: string (nullable = true)
 |-- llama_generated_abstract: string (nullable = true)
 |-- openai_generated_abstract: string (nullable = true)
 |-- method: string (nullable = true)


 Sample
+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+---------+
|                       original_abstract|                allam_generated_abstract|                 jais_generated_abstract|                llama_generated_abstract|               openai_generated_abstract|   method|
+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+---------+
|كثيرا ما ارتبطت

Prepare human vs AI data and show label distribution

In [11]:

human_df = df.select(
    col("original_abstract").alias("text"),
    col("method")
).withColumn("label", lit("human"))

# 2. Extract AI generated abstracts
ai_df = df.select(
    col("allam_generated_abstract").alias("text"),
    col("method")
).withColumn("label", lit("ai"))

# 3. Combine and display distribution
combined_df = human_df.unionByName(ai_df)

print("Label Distribution ")
combined_df.groupBy("label").count().show()

Label Distribution 
+-----+-----+
|label|count|
+-----+-----+
|human|18191|
|   ai|18191|
+-----+-----+

